# EP07 — Self-Supervised Learning
**COMPSCI 713 · S1 2024 Q7 · 16 marks**

| Part | Topic | Marks |
|------|-------|-------|
| a | Pretext and downstream tasks | 8 |
| b-i | How SSL differs from Supervised Learning | 4 |
| b-ii | How SSL differs from Unsupervised Learning | 4 |

---

## Part a — Pretext and Downstream Tasks [8 marks]

### Pretext Task
A **pretext task** is an artificially constructed supervised task whose labels are derived *automatically from the data itself* — no human annotation required.

The goal of the pretext task is NOT the task itself. It is to force the model to learn useful, general-purpose representations of the data.

| Domain | Example pretext tasks |
|--------|----------------------|
| NLP | Predict next word (GPT); predict masked words (BERT); predict sentence order |
| Vision | Predict image rotation angle; solve jigsaw puzzle; reconstruct masked patches (MAE) |
| Audio | Predict future audio frames; reconstruct masked spectrograms |
| Contrastive | Two augmented views of the same sample should be similar (SimCLR, MoCo) |

### Downstream Task
The **downstream task** is the actual task of interest, which typically has limited labelled data.

After pretext training, the encoder's learned representations are *transferred* to the downstream task. The model is then fine-tuned (or linearly probed) with a small labelled dataset.

Examples:
- BERT (pretrained on masked LM) -> fine-tuned on sentiment, NER, QA
- MAE (pretrained on image reconstruction) -> fine-tuned on ImageNet classification
- SimCLR (pretrained on contrastive pairs) -> fine-tuned on object recognition

**Key insight:** pretext tasks provide a free training signal from vast unlabelled data, bootstrapping powerful representations that transfer to data-scarce downstream tasks.

---

## Part b — Comparisons [8 marks]

### b-i: SSL vs Supervised Learning

| Aspect | Supervised Learning | Self-Supervised Learning |
|--------|--------------------|--------------------------|
| Labels | Human-annotated (expensive) | Auto-generated from data structure |
| Scale | Limited by annotation budget | Can use internet-scale unlabelled data |
| Generalisation | Task-specific | Representations transfer across many tasks |
| Training signal | External (human labels) | Internal (data structure itself) |

Key distinction: SSL is trained *without human labels* but still uses a supervised-style loss; the supervision comes from the structure of the data itself.

### b-ii: SSL vs Unsupervised Learning

| Aspect | Unsupervised Learning | Self-Supervised Learning |
|--------|----------------------|--------------------------|
| Goal | Find structure/density in data | Learn transferable representations |
| Loss | No fixed target (e.g., likelihood, clustering) | Predictive target auto-derived from data |
| Transfer quality | Features may not transfer well | Explicitly designed to produce transferable features |
| Examples | K-means, PCA, VAE, GMM | BERT, GPT, SimCLR, MAE, DINO |

Key distinction: both avoid human labels, but SSL defines a *specific predictive objective* designed to encourage rich, transferable representations, whereas unsupervised methods model the data distribution more generally.

---

## Code Demo — SimCLR-style Contrastive Self-Supervised Learning

In [ ]:
import torch, torch.nn as nn, torch.nn.functional as F
import numpy as np, matplotlib.pyplot as plt
from sklearn.decomposition import PCA

torch.manual_seed(42); np.random.seed(42)

# ─── Synthetic data: 4 classes in 16-dim space (labels HIDDEN during pretext) ─
N_CLASS, N_PER, DIM = 4, 50, 16
means = torch.randn(N_CLASS, DIM) * 4
X_list, y_list = [], []
for c in range(N_CLASS):
    X_list.append(means[c] + torch.randn(N_PER, DIM) * 0.8)
    y_list += [c]*N_PER
X = torch.cat(X_list)

def augment(x, noise=0.3): return x + torch.randn_like(x) * noise

class Encoder(nn.Module):
    def __init__(self): super().__init__()
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(DIM, 64), nn.ReLU(),
            nn.Linear(64, 32), nn.ReLU(),
            nn.Linear(32, 8)
        )
    def forward(self, x): return F.normalize(self.net(x), dim=-1)

def nt_xent(z1, z2, tau=0.5):
    'Normalised temperature-scaled cross-entropy (SimCLR loss).'
    N = z1.size(0)
    z = torch.cat([z1, z2])
    sim = torch.mm(z, z.T) / tau
    sim.fill_diagonal_(-1e9)
    labels = torch.cat([torch.arange(N)+N, torch.arange(N)])
    return F.cross_entropy(sim, labels)

enc = Encoder()
opt = torch.optim.Adam(enc.parameters(), lr=3e-3)
losses = []

# ── Pretext training — NO class labels used ───────────────────────────────────
for ep in range(300):
    z1 = enc(augment(X))
    z2 = enc(augment(X))
    loss = nt_xent(z1, z2)
    opt.zero_grad(); loss.backward(); opt.step()
    if ep % 50 == 0: losses.append(loss.item())

print('Pretext losses:', [f'{l:.3f}' for l in losses])

# ── Downstream: visualise learnt embeddings (labels only for colouring) ────────
with torch.no_grad(): emb = enc(X).numpy()
pca = PCA(2).fit_transform(emb)
y_arr = np.array(y_list)

fig, axes = plt.subplots(1, 2, figsize=(11,4))
c = plt.cm.tab10(y_arr/N_CLASS)
axes[0].scatter(X[:,0].numpy(), X[:,1].numpy(), c=c, s=18)
axes[0].set_title('Raw input (dim 0 vs 1 of 16)')
axes[1].scatter(pca[:,0], pca[:,1], c=c, s=18)
axes[1].set_title('SSL embeddings (PCA to 2D)\nClasses cluster WITHOUT labels')
for ax in axes: ax.set_xlabel('Dim 1'); ax.set_ylabel('Dim 2')
plt.suptitle('Self-supervised contrastive learning\n'
             '(pretext: make augmented views similar)', y=1.01)
plt.tight_layout(); plt.show()